## Imports

In [ ]:
import os
import pathlib
import re
import time
import shutil
from pathlib import Path

import mne
import pandas as pd
from tqdm import tqdm

## Set current dir to project root dir

In [ ]:
def find_project_root():
    """Walk up from CWD until we find the project root."""
    markers = [".git", "Makefile", "renv.lock", ".Rprofile"]
    path = Path.cwd()
    while path != path.parent:
        if any((path / m).exists() for m in markers):
            return path
        path = path.parent
    raise FileNotFoundError("Could not find project root")

os.chdir(find_project_root())

## Helper functions

### `format_elapsed(seconds)`

Formats an elapsed time in seconds into a human-readable string following the
[NIST Guide to the SI, Chapter 7](https://www.nist.gov/pml/special-publication-811/nist-guide-si-chapter-7-rules-and-style-conventions-expressing-values)
conventions for expressing values of quantities with units.

Examples: `41.3 s`, `1 min 41.3 s`, `1 h 20 min 10.6 s`

In [ ]:
def format_elapsed(seconds):
    if seconds >= 3600:
        h = int(seconds // 3600)
        m = int((seconds % 3600) // 60)
        s = seconds % 60
        return f"{h} h {m} min {s:.1f} s"
    elif seconds >= 60:
        m = int(seconds // 60)
        s = seconds % 60
        return f"{m} min {s:.1f} s"
    else:
        return f"{seconds:.1f} s"

## Functions

In [ ]:
def get_subject_folders(parent_directory):
    """
    Returns a list of paths for all subfolders
    starting with 'sub-' in the given directory.
    """
    path = Path(parent_directory)

    # .glob('sub-*') looks for items starting with 'sub-'
    # is_dir() ensures we only get folders, not files
    subject_folders = [str(f) for f in path.glob('sub-*') if f.is_dir()]

    return sorted(subject_folders)

# Example Usage:
# folders = get_subject_folders('/path/to/your/eeg_data')
# print(f"Found {len(folders)} subject folders.")

In [ ]:
def extract_unique_stimuli(file_path):
    """
    Parses a BrainVision .vmrk file and returns a sorted list
    of all unique stimulus descriptions (trigger codes).
    """
    stimuli = set()

    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            for line in file:
                # Look for lines starting with Mk followed by digits (e.g., Mk100=...)
                if line.startswith('Mk'):
                    # Split by '=' to separate key and values, then split values by ','
                    try:
                        # Format: Mk<Num>=Type,Description,Position,Size,Channel
                        content = line.split('=')[1]
                        parts = content.split(',')

                        marker_type = parts[0].strip()
                        description = parts[1].strip()

                        # Only add if the type is exactly 'Stimulus'
                        if marker_type == 'Stimulus':
                            stimuli.add(description)
                    except (IndexError, ValueError):
                        # Skip malformed lines
                        continue

    except FileNotFoundError:
        return "Error: File not found."

    # Return as a sorted list for easier viewing
    return sorted(list(stimuli))

## Variables

## Main

In [ ]:
main_data_folder = "./ds006018"
tasks = ["task-auditoryoddball", "task-flanker", "task-visualoddball", "task-visualsearch"]

In [ ]:
paths_to_original_data_actors = get_subject_folders(main_data_folder)

In [ ]:
# ── Timer start ───────────────────────────────────────────────────────────────────────
start = time.time()
# ──────────────────────────────────────────────────────────────────────────────────────

for i in tqdm(range(len(paths_to_original_data_actors))):

    for task_no in range(len(tasks)):

        task = tasks[task_no]

        sub_number = paths_to_original_data_actors[i].split('/')[-1]

        path_to_vhdr = paths_to_original_data_actors[i] + "/eeg/" + sub_number + "_" + task + "_eeg.vhdr"
        path_to_vmrk = paths_to_original_data_actors[i] + "/eeg/" + sub_number + "_" + task + "_eeg.vmrk"

        directory = Path("./ds006018_per_stimuli/"+sub_number)
        directory.mkdir(parents=True, exist_ok=True)

        file_path = Path(path_to_vhdr)

        if file_path.is_file():
            print("The file exists!")



            # Get unique stimulus numbers from the .vmrk file
            unique_stimuli_numers = extract_unique_stimuli(path_to_vmrk)
            print(unique_stimuli_numers)


            # 1. Load the BrainVision data
            raw = mne.io.read_raw_brainvision(path_to_vhdr, preload=True)


            # 2. Set the montage (Standard 10-20 system for electrode locations)
            montage = mne.channels.make_standard_montage('standard_1020')
            raw.set_montage(montage)


            # 3. Filtering (Standard for EEG: 0.1Hz to 40Hz)
            raw.filter(l_freq=0.1, h_freq=40.0)


            # 4. Plotting the data to inspect for noise
            #raw.plot(n_channels=15, duration=5, scalings='auto')


            # 5. Preprocessing (Required for FDA to reduce noise)
            #raw.resample(200)               # Downsample to reduce R processing time - let's do this since we have a lot of data and FDA can be computationally intensive

            # 6. Create Epochs (FDA usually analyzes trials/segments) - by
            # This assumes you have event markers in your .vmrk file
            events, event_id = mne.events_from_annotations(raw)

            stimulus_dataframes = {}

            # Iterate through every stimulus
            for event_name, event_val in event_id.items():
                # Clean the name for filenames (e.g., 'Stimulus_S1')
                clean_name = event_name.replace('/', '_').replace(' ', '')

                try:
                    # Pass a dictionary where the key is the name and value is the integer ID
                    # This resolves the "must be an int, got str" error
                    current_epochs = mne.Epochs(raw, events, event_id={event_name: event_val},
                                                tmin=-0.2, tmax=0.8, preload=True)

                    if len(current_epochs) > 0:
                        df_temp = current_epochs.to_data_frame()
                        # Drop 'condition' column and standardise column order to match R/eegUtils output
                        df_temp = df_temp.drop(columns=["condition"], errors="ignore")
                        df_temp = df_temp[["time", "epoch"] + [c for c in df_temp.columns if c not in ["time", "epoch"]]]

                        # Store and export
                        stimulus_dataframes[clean_name] = df_temp
                        df_temp.to_csv("./ds006018_per_stimuli/"+sub_number+"/"+task+"_"+clean_name+".csv", index=False)

                        print(f"Successfully created: eeg_{clean_name}.csv ({len(current_epochs)} trials)")

                except Exception as e:
                    print(f"Skipping {event_name}: {e}")

        else:
            print(path_to_vhdr+"File not found.")

# ── Timer end ─────────────────────────────────────────────────────────────────────────
print("───────────────────────────────────────────────────────────────────────────────")
print(f"  Time elapsed: {format_elapsed(time.time() - start)}")
print("───────────────────────────────────────────────────────────────────────────────")
# ──────────────────────────────────────────────────────────────────────────────────────